# 21 - Backend Comparison

Compare the local simulator with the Aer backend.

**Concepts:** Backend routing, qs.run, performance, consistency

In [ ]:
import quantsdk as qs
import time

## Local Simulator

In [ ]:
circuit = qs.Circuit(4, name="test")
circuit.h(0).cx(0, 1).cx(1, 2).cx(2, 3).measure_all()

# Run on local simulator (default)
t0 = time.perf_counter()
result_local = qs.run(circuit, shots=4000, seed=42, backend="local")
t_local = time.perf_counter() - t0

print("Local Simulator Results:")
print(f"  Time: {t_local*1000:.1f} ms")
print(f"  Counts: {result_local.counts}")
print(f"  Most likely: {result_local.most_likely}")

## Aer Backend (if available)

In [ ]:
try:
    t0 = time.perf_counter()
    result_aer = qs.run(circuit, shots=4000, seed=42, backend="aer")
    t_aer = time.perf_counter() - t0
    
    print("Aer Backend Results:")
    print(f"  Time: {t_aer*1000:.1f} ms")
    print(f"  Counts: {result_aer.counts}")
    print(f"  Most likely: {result_aer.most_likely}")
except Exception as e:
    print(f"Aer backend not available: {e}")
    print("Install: pip install 'quantsdk[ibm]'")

## Consistency Check

In [ ]:
# Both backends should give consistent results for simple circuits
test_circuits = [
    ("H|0>", qs.Circuit(1).h(0).measure(0)),
    ("Bell", qs.Circuit(2).h(0).cx(0, 1).measure_all()),
    ("GHZ-3", qs.Circuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()),
]

for name, circ in test_circuits:
    r = qs.run(circ, shots=4000, seed=42, backend="local")
    print(f"{name:8s}: {r.most_likely} | top_2={r.top_k(2)}")

## Scaling Benchmark

In [ ]:
# How does the local simulator scale with qubit count?
for n in [4, 8, 12, 16, 20]:
    circuit = qs.Circuit(n)
    circuit.h(0)
    for i in range(n - 1):
        circuit.cx(i, i + 1)
    circuit.measure_all()
    
    t0 = time.perf_counter()
    result = qs.run(circuit, shots=1000, seed=42, backend="local")
    elapsed = time.perf_counter() - t0
    
    print(f"n={n:2d}: {elapsed*1000:8.1f} ms | {result.most_likely[:8]}... | states=2^{n}={2**n}")